# 00 — JUMP-CP Compound Registry

Central compound list that all database mapping notebooks start from.
Outputs `cache/jump_compounds.parquet` with plate-type annotation.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("../..")  # repo root
CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)

In [ ]:
# Load JUMP compound registry
compounds = pd.read_csv(ROOT / "inputs/metadata/compound.csv.gz")
print(f"Total compounds: {len(compounds):,}")
print(f"Unique InChIKeys: {compounds['Metadata_InChIKey'].nunique():,}")
compounds.head()

In [ ]:
# Annotate plate type: TARGET2 (302 shared tool compounds) vs COMPOUND
# TARGET2 compounds are the JCP2022 IDs that appear in the TARGET2 plate metadata
well = pd.read_csv(ROOT / "inputs/metadata/well.csv.gz")
target2_jcps = well.loc[well["Metadata_PlateType"] == "TARGET2", "Metadata_JCP2022"].unique()

compounds["plate_type"] = np.where(
    compounds["Metadata_JCP2022"].isin(target2_jcps), "TARGET2", "COMPOUND"
)
print(compounds["plate_type"].value_counts())
print(f"\nTARGET2 compounds: {(compounds['plate_type'] == 'TARGET2').sum()}")
print(f"COMPOUND compounds: {(compounds['plate_type'] == 'COMPOUND').sum()}")

In [ ]:
# Save
compounds.to_parquet(CACHE / "jump_compounds.parquet", index=False)
print(f"Saved {len(compounds):,} compounds to cache/jump_compounds.parquet")